In [123]:
import pyodbc
import pandas as pd
import numpy as np

print(pyodbc.drivers())

['SQL Server', 'SQL Server Native Client 11.0', 'B1CRHPROXY', 'ODBC Driver 13 for SQL Server', 'Microsoft Access Driver (*.mdb, *.accdb)', 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)', 'Microsoft Access Text Driver (*.txt, *.csv)', 'Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)', 'ODBC Driver 18 for SQL Server', 'SQL Server Native Client RDA 11.0', 'ODBC Driver 17 for SQL Server']


In [124]:
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus
from dotenv import load_dotenv
import os
import pyodbc

load_dotenv()

SERVER = os.getenv("DB_SERVER1")
DBNAME = os.getenv("DB_NAME1")
USER = os.getenv("DB_USER1")
PASS = os.getenv("DB_PASSWORD1")
ENCRYPT = os.getenv("DB_ENCRYPT", "yes")
TRUST = os.getenv("DB_TRUST_CERT", "yes")

SERVER = SERVER.replace("\\\\", "\\")  # normaliza


cnx = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    f"SERVER={SERVER};"
    f"DATABASE={DBNAME};"
    f"UID={USER};"
    f"PWD={PASS};"
    f"Encrypt={ENCRYPT};TrustServerCertificate={TRUST};"
)

engine = create_engine(f"mssql+pyodbc:///?odbc_connect={quote_plus(cnx)}")

In [125]:
from sqlalchemy import text
try:
    with engine.connect() as conn:
        print("Ping:", conn.execute(text("SELECT 1")).scalar_one())
        print("Version:", conn.execute(text("SELECT @@VERSION")).scalar_one())
        print("✅ Conexión OK")
except Exception as e:
    print("❌ Error al conectar:", e)

Ping: 1
Version: Microsoft SQL Server 2017 (RTM) - 14.0.1000.169 (X64) 
	Aug 22 2017 17:04:49 
	Copyright (C) 2017 Microsoft Corporation
	Standard Edition (64-bit) on Windows Server 2019 Standard 10.0 <X64> (Build 17763: )

✅ Conexión OK


In [126]:
import pandas as pd
# 🔥 TEST 2: traer datos a pandas
query = "SELECT TOP 10 * FROM OITM"

df = pd.read_sql(query, engine)

df.head(5)

,ItemCode,ItemName,FrgnName,ItmsGrpCod,CstGrpCode,VatGourpSa,CodeBars,VATLiable,PrchseItem,SellItem,...,U_CI_15_Familia,U_CI_16_CaMeHP,U_CI_16_Ratio,U_CI_16_DiEjeSal,U_CI_14_DiExtRoMM,U_CI_14_NoCotCo,U_CI_14_AnchoMM,U_CI_14_SentR,U_CI_14_DiEjeIN,U_CI_14_TiRod
0,"# 111A-30"" X 1.5",NaN,NaN,100,-1,NaN,None,Y,Y,Y,...,None,None,None,None,None,None,None,None,None,None
1,"# 111A-30"" X 1.5""",BANDA TRANSPORTADORA CON ESPUMA Y GUIA,160,106,-1,,None,Y,Y,Y,...,None,None,None,None,None,None,None,None,None,None
2,"# 111A-54"" X 1.5",NaN,NaN,100,-1,NaN,None,Y,Y,Y,...,None,None,None,None,None,None,None,None,None,None
3,"# 111A-54"" X 1.5""",BANDA TRANSPORTADORA CON ESPUMA Y GUIA,160,106,-1,,None,Y,Y,Y,...,None,None,None,None,None,None,None,None,None,None
4,#108GL,BANDA TRANSPORTADORA 2PLY POLYCR57 WHITE PU GL...,160,106,-1,,None,Y,Y,Y,...,None,None,None,None,None,None,None,None,None,None


In [127]:
Query_OITM = """
SELECT
    /* =====================================================
       IDENTIFICACIÓN DEL ARTÍCULO
       ===================================================== */

    T0.ItemCode AS [Código de artículo],
    T0.ItemName AS [Descripción completa],
    TG.ItmsGrpNam AS [Grupo de artículo],
    T0.InvntryUom AS [Unidad de medida de inventario],
    TM.FirmName AS [Marca / Fabricante],


    /* =====================================================
       COMPRAS
       ===================================================== */

    T0.LastPurDat AS [Última fecha de compra],
    T0.LastPurPrc AS [Último precio de compra],


    /* =====================================================
       COSTOS
       ===================================================== */

    TW.AvgPrice AS [Costo promedio],


    /* =====================================================
       INVENTARIO POR BODEGA
       ===================================================== */

    TW.WhsCode AS [Código de bodega],
    TW.OnHand AS [Stock en mano por bodega],
    TW.IsCommited AS [Stock comprometido],
    TW.OnOrder AS [Stock en pedido / tránsito],


    /* =====================================================
       PARÁMETROS DE COMPRA E INVENTARIO
       ===================================================== */

    T0.MinOrdrQty AS [Cantidad mínima de compra],
    TW.MinStock AS [Stock mínimo],
    TW.MaxStock AS [Stock máximo],


    /* =====================================================
       LISTAS DE PRECIO COMO COLUMNAS
       ===================================================== */

    MAX(CASE WHEN TP.PriceList = 11 THEN TP.Price END) AS [Precio lista 11 - Promociones ecommerce],
    MAX(CASE WHEN TP.PriceList = 12 THEN TP.Price END) AS [Precio lista 12 - B2B VIP],
    MAX(CASE WHEN TP.PriceList = 13 THEN TP.Price END) AS [Precio lista 13 - B2C],
    MAX(CASE WHEN TP.PriceList = 14 THEN TP.Price END) AS [Precio lista 14 - B2B estándar]

FROM OITM T0

LEFT JOIN OITB TG
    ON T0.ItmsGrpCod = TG.ItmsGrpCod

LEFT JOIN OMRC TM
    ON T0.FirmCode = TM.FirmCode

INNER JOIN OITW TW
    ON T0.ItemCode = TW.ItemCode

LEFT JOIN ITM1 TP
    ON T0.ItemCode = TP.ItemCode
    AND TP.PriceList IN (11, 12, 13, 14)

WHERE
    T0.SellItem = 'Y'
    AND T0.PrchseItem = 'Y'
    AND T0.InvntItem = 'Y'
    AND T0.validFor = 'Y'
    AND T0.frozenFor = 'N'

    AND TW.WhsCode IN ('11', '14')
    AND TW.OnHand > 0
    AND TM.FirmName LIKE '%SIEMENS%'

GROUP BY
    T0.ItemCode,
    T0.ItemName,
    TG.ItmsGrpNam,
    T0.InvntryUom,
    TM.FirmName,

    T0.LastPurDat,
    T0.LastPurPrc,

    TW.AvgPrice,
    TW.WhsCode,
    TW.OnHand,
    TW.IsCommited,
    TW.OnOrder,

    T0.MinOrdrQty,
    TW.MinStock,
    TW.MaxStock

ORDER BY
    T0.ItemCode;
"""

In [128]:
df_OITM= pd.read_sql(Query_OITM, engine)
df_OITM.info()


<class 'pandas.DataFrame'>
RangeIndex: 383 entries, 0 to 382
Data columns (total 19 columns):
 #   Column                                   Non-Null Count  Dtype         
---  ------                                   --------------  -----         
 0   Código de artículo                       383 non-null    str           
 1   Descripción completa                     383 non-null    str           
 2   Grupo de artículo                        383 non-null    str           
 3   Unidad de medida de inventario           383 non-null    str           
 4   Marca / Fabricante                       383 non-null    str           
 5   Última fecha de compra                   383 non-null    datetime64[us]
 6   Último precio de compra                  383 non-null    float64       
 7   Costo promedio                           383 non-null    float64       
 8   Código de bodega                         383 non-null    str           
 9   Stock en mano por bodega                 383 non-null 

In [129]:
ruta_campos_OITM_SAP="./Datos/campos_OITM_SAPB1.csv"
df_OITM.to_csv("articulos_siemens.csv",
    index=False,
    encoding="utf-8-sig",
    sep=";"
)

## Ventas netas mensuales y totales anuales

En este bloque se agregan al query base las ventas históricas de cada artículo desde enero de 2022 hasta la fecha actual.

Las ventas se calculan como venta neta real, considerando facturas de venta en positivo y notas de crédito en negativo. Esto permite descontar devoluciones o ajustes comerciales y obtener una cantidad vendida más representativa.

Las columnas mensuales se generan dinámicamente desde Python, de modo que cada mes aparezca como una columna independiente en el DataFrame final. Además, se agregan columnas de total anual para cada año, incluyendo el año actual aunque todavía no esté cerrado.

Esta información servirá para analizar rotación, comportamiento de demanda, estacionalidad y relación entre inventario disponible y ventas históricas.

In [130]:
from datetime import date

# ============================================================
# CONFIGURACIÓN DE FECHAS
# ============================================================

fecha_inicio = date(2024, 1, 1)
fecha_fin = date.today()

meses_es = {
    1: "Enero",
    2: "Febrero",
    3: "Marzo",
    4: "Abril",
    5: "Mayo",
    6: "Junio",
    7: "Julio",
    8: "Agosto",
    9: "Septiembre",
    10: "Octubre",
    11: "Noviembre",
    12: "Diciembre"
}


# ============================================================
# GENERAR COLUMNAS MENSUALES Y TOTALES ANUALES
# Formato requerido:
# Enero_2022, Febrero_2022, ..., Diciembre_2022, 2022
# Enero_2023, Febrero_2023, ..., Diciembre_2023, 2023
# ============================================================

columnas_ventas_sql = []
columnas_ventas_finales = []

anio_inicio = fecha_inicio.year
anio_fin = fecha_fin.year

for anio in range(anio_inicio, anio_fin + 1):

    # --------------------------------------------------------
    # Columnas mensuales del año
    # --------------------------------------------------------
    for mes in range(1, 13):

        # No generar meses futuros del año actual
        if anio == fecha_fin.year and mes > fecha_fin.month:
            break

        nombre_mes = meses_es[mes]
        nombre_columna = f"{nombre_mes}_{anio}"

        fecha_mes_inicio = f"{anio}-{mes:02d}-01"

        if mes == 12:
            fecha_mes_fin = f"{anio + 1}-01-01"
        else:
            fecha_mes_fin = f"{anio}-{mes + 1:02d}-01"

        columna_sql = f"""
        SUM(
            CASE 
                WHEN DocDate >= '{fecha_mes_inicio}' 
                 AND DocDate <  '{fecha_mes_fin}'
                THEN CantidadNeta 
                ELSE 0 
            END
        ) AS [{nombre_columna}]
        """

        columnas_ventas_sql.append(columna_sql)

        columnas_ventas_finales.append(
            f"ISNULL(VP.[{nombre_columna}], 0) AS [{nombre_columna}]"
        )

    # --------------------------------------------------------
    # Columna total anual
    # Va después del último mes generado de cada año.
    # Para años cerrados suma enero-diciembre.
    # Para el año actual suma desde enero hasta la fecha actual.
    # --------------------------------------------------------

    nombre_columna_anual = str(anio)

    columna_anual_sql = f"""
        SUM(
            CASE 
                WHEN DocDate >= '{anio}-01-01' 
                 AND DocDate <  '{anio + 1}-01-01'
                THEN CantidadNeta 
                ELSE 0 
            END
        ) AS [{nombre_columna_anual}]
    """

    columnas_ventas_sql.append(columna_anual_sql)

    columnas_ventas_finales.append(
        f"ISNULL(VP.[{nombre_columna_anual}], 0) AS [{nombre_columna_anual}]"
    )


# ============================================================
# UNIR COLUMNAS PARA INSERTARLAS EN EL QUERY SQL
# ============================================================

columnas_ventas_sql = ",\n".join(columnas_ventas_sql)

columnas_ventas_finales_sql = ",\n    ".join(columnas_ventas_finales)

In [131]:
Query_OITM_Ventas= f"""
WITH VentasBase AS (

    /* =====================================================
       FACTURAS DE VENTA
       OINV = encabezado de factura
       INV1 = detalle de factura

       Las cantidades de factura se toman como positivas.
       ===================================================== */

    SELECT
        L.ItemCode,
        H.DocDate,
        L.Quantity AS CantidadNeta
    FROM OINV H
    INNER JOIN INV1 L
        ON H.DocEntry = L.DocEntry
    WHERE
        H.CANCELED = 'N'
        AND H.DocDate >= '2022-01-01'
        AND H.DocDate < DATEADD(DAY, 1, CAST(GETDATE() AS date))


    UNION ALL


    /* =====================================================
       NOTAS DE CRÉDITO
       ORIN = encabezado de nota de crédito
       RIN1 = detalle de nota de crédito

       Las cantidades de nota de crédito se toman como negativas
       para calcular venta neta real.
       ===================================================== */

    SELECT
        L.ItemCode,
        H.DocDate,
        L.Quantity * -1 AS CantidadNeta
    FROM ORIN H
    INNER JOIN RIN1 L
        ON H.DocEntry = L.DocEntry
    WHERE
        H.CANCELED = 'N'
        AND H.DocDate >= '2022-01-01'
        AND H.DocDate < DATEADD(DAY, 1, CAST(GETDATE() AS date))
),


VentasPivot AS (

    /* =====================================================
       CONSOLIDACIÓN DE VENTAS NETAS POR ARTÍCULO

       Aquí se transforman las ventas mensuales en columnas.
       También se agregan los totalizadores anuales.
       ===================================================== */

    SELECT
        ItemCode,
        {columnas_ventas_sql}
    FROM VentasBase
    GROUP BY
        ItemCode
),


PreciosPivot AS (

    /* =====================================================
       LISTAS DE PRECIO COMO COLUMNAS

       ITM1 guarda un precio por artículo y por lista.
       Aquí convertimos las listas 11, 12, 13 y 14 en columnas.
       ===================================================== */

    SELECT
        ItemCode,
        MAX(CASE WHEN PriceList = 11 THEN Price END) AS [Precio lista 11 - Promociones ecommerce],
        MAX(CASE WHEN PriceList = 12 THEN Price END) AS [Precio lista 12 - B2B VIP],
        MAX(CASE WHEN PriceList = 13 THEN Price END) AS [Precio lista 13 - B2C],
        MAX(CASE WHEN PriceList = 14 THEN Price END) AS [Precio lista 14 - B2B estándar]
    FROM ITM1
    WHERE
        PriceList IN (11, 12, 13, 14)
    GROUP BY
        ItemCode
),


InventarioFiltrado AS (

    /* =====================================================
       INVENTARIO FILTRADO

       Se consideran únicamente bodegas 11 y 14 con stock positivo.

       Este bloque consolida el inventario por artículo para evitar
       duplicidades en caso de que SAP tenga registros en más de una
       bodega. Si el artículo solo tiene stock en una bodega, el resultado
       será exactamente esa bodega.
       ===================================================== */

    SELECT
        ItemCode,
        MAX(WhsCode) AS WhsCode,
        SUM(OnHand) AS OnHand,
        SUM(IsCommited) AS IsCommited,
        SUM(OnOrder) AS OnOrder,
        MAX(AvgPrice) AS AvgPrice,
        MAX(MinStock) AS MinStock,
        MAX(MaxStock) AS MaxStock
    FROM OITW
    WHERE
        WhsCode IN ('11', '14')
        AND OnHand > 0
    GROUP BY
        ItemCode
)


SELECT
    /* =====================================================
       IDENTIFICACIÓN DEL ARTÍCULO
       ===================================================== */

    T0.ItemCode AS [Código de artículo],
    T0.ItemName AS [Descripción completa],
    TG.ItmsGrpNam AS [Grupo de artículo],

    /* =====================================================
       CLASIFICACIÓN DEFINIDA POR EL USUARIO
       Campos personalizados creados en SAP B1 sobre OITM.
       Se usan para conservar la clasificación interna del artículo.
       ===================================================== */

    T0.U_CI_CAT AS [Categoría],
    T0.U_CI_SCAT AS [Subcategoría],
    T0.U_CI_TIPO AS [Tipo],

    T0.InvntryUom AS [Unidad de medida de inventario],
    TM.FirmName AS [Marca / Fabricante],


    /* =====================================================
       COMPRAS
       ===================================================== */

    T0.LastPurDat AS [Última fecha de compra],
    T0.LastPurPrc AS [Último precio de compra],


    /* =====================================================
       COSTOS
       ===================================================== */

    INV.AvgPrice AS [Costo promedio],


    /* =====================================================
       INVENTARIO
       ===================================================== */

    INV.WhsCode AS [Código de bodega],
    INV.OnHand AS [Stock en mano],
    INV.IsCommited AS [Stock comprometido],
    INV.OnOrder AS [Stock en pedido / tránsito],


    /* =====================================================
       PARÁMETROS DE COMPRA E INVENTARIO
       ===================================================== */

    T0.MinOrdrQty AS [Cantidad mínima de compra],
    INV.MinStock AS [Stock mínimo],
    INV.MaxStock AS [Stock máximo],


    /* =====================================================
       PRECIOS DE VENTA
       ===================================================== */

    PP.[Precio lista 11 - Promociones ecommerce],
    PP.[Precio lista 12 - B2B VIP],
    PP.[Precio lista 13 - B2C],
    PP.[Precio lista 14 - B2B estándar],


    /* =====================================================
       VENTAS NETAS MENSUALES Y TOTALIZADORES ANUALES

       Las columnas quedan en formato:
       Enero_2022, Febrero_2022, ..., Diciembre_2022, 2022
       ===================================================== */

    {columnas_ventas_finales_sql}

FROM OITM T0

LEFT JOIN OITB TG
    ON T0.ItmsGrpCod = TG.ItmsGrpCod

LEFT JOIN OMRC TM
    ON T0.FirmCode = TM.FirmCode

INNER JOIN InventarioFiltrado INV
    ON T0.ItemCode = INV.ItemCode

LEFT JOIN PreciosPivot PP
    ON T0.ItemCode = PP.ItemCode

LEFT JOIN VentasPivot VP
    ON T0.ItemCode = VP.ItemCode

WHERE
    /* =====================================================
       FILTROS DE ESTADO DEL ARTÍCULO

       Estos campos se usan únicamente como filtros.
       No se muestran como columnas en el resultado final.
       ===================================================== */

    T0.SellItem = 'Y'
    AND T0.PrchseItem = 'Y'
    AND T0.InvntItem = 'Y'
    AND T0.validFor = 'Y'
    AND T0.frozenFor = 'N'


    /* =====================================================
       FILTRO DE MARCA
       Solo artículos SIEMENS
       ===================================================== */

    AND TM.FirmName LIKE '%SIEMENS%'

ORDER BY
    T0.ItemCode;
"""


In [132]:
df_OITM_Ventas= pd.read_sql(Query_OITM_Ventas, engine)
df_OITM_Ventas.info()


<class 'pandas.DataFrame'>
RangeIndex: 383 entries, 0 to 382
Data columns (total 54 columns):
 #   Column                                   Non-Null Count  Dtype         
---  ------                                   --------------  -----         
 0   Código de artículo                       383 non-null    str           
 1   Descripción completa                     383 non-null    str           
 2   Grupo de artículo                        383 non-null    str           
 3   Categoría                                354 non-null    str           
 4   Subcategoría                             354 non-null    str           
 5   Tipo                                     337 non-null    str           
 6   Unidad de medida de inventario           383 non-null    str           
 7   Marca / Fabricante                       383 non-null    str           
 8   Última fecha de compra                   383 non-null    datetime64[us]
 9   Último precio de compra                  383 non-null 

In [133]:
import pandas as pd

# ============================================================
# AGREGAR COLUMNAS MANUALES AL DATAFRAME
# ============================================================
# Estas columnas no existen en SAP Business One.
# Se agregan al DataFrame como campos internos de análisis,
# revisión, clasificación o parametrización posterior.

columnas_adicionales = [
    "U_Ciclo_Vida",
    "U_Criticidad",
    "U_Pais_Origen",
    "U_Tipo_Abastecimiento",
    "U_Punto_Reorden",
    "U_Politica_Reposicion",
    "U_Aplica_Forecast",
    "U_Modelo_Forecast",
    "U_Tipo_Demanda",
    "U_Comentario_Rev",
    "U_Fecha_Revision",
    "U_Revisado_Por",
    "U_Canal_Principal",
    "U_Venta_Proyecto"
]

for columna in columnas_adicionales:
    df_OITM_Ventas[columna] = pd.NA

In [134]:
df_OITM_Ventas.info()

<class 'pandas.DataFrame'>
RangeIndex: 383 entries, 0 to 382
Data columns (total 68 columns):
 #   Column                                   Non-Null Count  Dtype         
---  ------                                   --------------  -----         
 0   Código de artículo                       383 non-null    str           
 1   Descripción completa                     383 non-null    str           
 2   Grupo de artículo                        383 non-null    str           
 3   Categoría                                354 non-null    str           
 4   Subcategoría                             354 non-null    str           
 5   Tipo                                     337 non-null    str           
 6   Unidad de medida de inventario           383 non-null    str           
 7   Marca / Fabricante                       383 non-null    str           
 8   Última fecha de compra                   383 non-null    datetime64[us]
 9   Último precio de compra                  383 non-null 

In [135]:
# ============================================================
# NOMBRE DINÁMICO DEL ARCHIVO
# ============================================================

anio_inicio = fecha_inicio.year
anio_fin = fecha_fin.year

nombre_archivo = f"./Datos/articulos_siemens_con_ventas_{anio_inicio}_{anio_fin}.csv"

df_OITM_Ventas.to_csv(nombre_archivo,
    index=False,
    encoding="utf-8-sig",
    sep=";"
)


## Parametrización externa del ciclo de vida

El ciclo de vida del artículo no se extrae directamente desde SAP Business One, ya que corresponde a una regla de negocio definida a partir de la clasificación interna del producto.

Para mantener una trazabilidad clara, esta asignación se manejará en un script independiente. El script toma las combinaciones únicas de Categoría CI, Subcategoría CI y Tipo CI, genera una tabla de parametrización y permite asignar un ciclo de vida promedio en meses para cada combinación.

Posteriormente, esa parametrización se cruza nuevamente contra el DataFrame principal para completar el campo `U_Ciclo_Vida`. Este enfoque evita asignar valores artículo por artículo y permite mantener una lógica más escalable, consistente y fácil de revisar.

In [140]:
# ============================================================
# CARGAR ARCHIVO PARAMETRIZADO DE CICLO DE VIDA
# ============================================================

df_ciclo_vida = pd.read_excel("./Datos/parametrizacion_ciclo_vida.xlsx") # Se parametriza manualmente ver script parametrizar_ciclo_vida.py)
df_ciclo_vida.head(5)

,Categoría,Subcategoría,Tipo,U_Ciclo_Vida,Ciclo_Vida_Años,Criterio / Justificación,Fuentes consultadas
0,EQUIPO ELECTRICO & AUTOMATIZACION,AUTOMATOS,AUTOMATOS,216,18,18 años. Familia de automatización con obsoles...,https://support.industry.siemens.com/cs/docume...
1,EQUIPO ELECTRICO & AUTOMATIZACION,BOTONERIA,CAJAS,144,12,12 años. Cajas de botonera: componente pasivo/...,https://support.industry.siemens.com/cs/docume...
2,EQUIPO ELECTRICO & AUTOMATIZACION,BOTONERIA,"INDICADORES, POTENCIOMETROS & OTROS",120,10,"10 años. Indicadores, potenciómetros y accesor...",https://support.industry.siemens.com/cs/docume...
3,EQUIPO ELECTRICO & AUTOMATIZACION,BOTONERIA,MANETAS,120,10,10 años. Manetas/selector: vida dominada por c...,https://support.industry.siemens.com/cs/docume...
4,EQUIPO ELECTRICO & AUTOMATIZACION,BOTONERIA,MODULOS,120,10,10 años. Módulos de botonera/contactos: desgas...,https://support.industry.siemens.com/cs/docume...


In [141]:
# ============================================================
# CONSERVAR SOLO LAS COLUMNAS DE CLASIFICACIÓN Y CICLO DE VIDA
# ============================================================
# La parametrización NO está por código de artículo.
# Está por combinación:
# Categoría CI + Subcategoría CI + Tipo CI

columnas_merge = [
    "Categoría",
    "Subcategoría",
    "Tipo"
]

df_ciclo_vida = df_ciclo_vida[
    columnas_merge + ["U_Ciclo_Vida"]
].drop_duplicates(subset=columnas_merge)

df_ciclo_vida.head(5)

,Categoría,Subcategoría,Tipo,U_Ciclo_Vida
0,EQUIPO ELECTRICO & AUTOMATIZACION,AUTOMATOS,AUTOMATOS,216
1,EQUIPO ELECTRICO & AUTOMATIZACION,BOTONERIA,CAJAS,144
2,EQUIPO ELECTRICO & AUTOMATIZACION,BOTONERIA,"INDICADORES, POTENCIOMETROS & OTROS",120
3,EQUIPO ELECTRICO & AUTOMATIZACION,BOTONERIA,MANETAS,120
4,EQUIPO ELECTRICO & AUTOMATIZACION,BOTONERIA,MODULOS,120


In [142]:
# ============================================================
# EVITAR DUPLICIDAD DE U_Ciclo_Vida
# ============================================================

if "U_Ciclo_Vida" in df_OITM_Ventas.columns:
    df_OITM_Ventas = df_OITM_Ventas.drop(columns=["U_Ciclo_Vida"])

In [143]:
# ============================================================
# MERGE FINAL
# ============================================================

df_OITM_Ventas = df_OITM_Ventas.merge(
    df_ciclo_vida,
    on=columnas_merge,
    how="left"
)


# ============================================================
# VALIDACIÓN RÁPIDA
# ============================================================

print("Artículos sin ciclo de vida asignado:")
print(df_OITM_Ventas["U_Ciclo_Vida"].isna().sum())

Artículos sin ciclo de vida asignado:
29


In [144]:
df_OITM_Ventas.to_csv(nombre_archivo,
    index=False,
    encoding="utf-8-sig",
    sep=";"
)
